In [ ]:
#@title 1. Configuración del Entorno
import os

# Clonar el repositorio si no existe
if not os.path.exists("Signal-Analysis-Vol-I"):
    !git clone https://github.com/ecundir/Signal-Analysis-Vol-I.git

# Cambiar a la carpeta del capítulo
%cd Signal-Analysis-Vol-I/capitulo_2
print("✅ Entorno configurado correctamente.")

No te limites a leer la teoría. Usa este Laboratorio Vivo y trata de configurar un repetidor que tenga una ganancia de 5, un retraso de 20ms, pero que sea Inestable. ¿Qué valor de feedback necesitas? Observa cómo la integridad de la señal desaparece cuando los axiomas de los sistemas LTI se rompen.

In [ ]:
#@title 📡 Centro de Control del Repetidor (Propiedades LTI) { run: "auto" }
#@markdown ---
#@markdown ### 1. Linealidad (Control de Ganancia)
ganancia = 2 #@param {type:"slider", min:1, max:10, step:0.5}
saturacion_on = False #@param {type:"boolean"}
#@markdown ---
#@markdown ### 2. Invarianza y Causalidad (El Canal)
delay_muestras = 50 #@param {type:"slider", min:0, max:200, step:10}
canal_variable = False #@param {type:"boolean"}
#@markdown ---
#@markdown ### 3. Estabilidad BIBO (Retroalimentación)
feedback_factor = 0 #@param {type:"slider", min:0, max:1.2, step:0.1}

import numpy as np
import matplotlib.pyplot as plt

# Generación de señal de entrada (Pulso de datos)
t = np.linspace(0, 1, 1000)
x = np.where((t > 0.1) & (t < 0.3), 1.0, 0.0)

# Inicialización de salida
y = np.zeros_like(x)

# Simulación del Operador H (Procesamiento paso a paso)
for n in range(len(x)):
    # CAUSALIDAD: El valor actual depende de x[n - delay]
    idx_delayed = max(0, n - delay_muestras)
    entrada_procesada = x[idx_delayed]

    # INVARIANZA: Si el canal varía, la ganancia cambia con el tiempo n
    g_efectiva = ganancia * (1 + 0.5*np.sin(2*np.pi*n/100)) if canal_variable else ganancia

    # LINEALIDAD: Aplicamos ganancia (y opcionalmente saturación/clipping)
    salida_temp = g_efectiva * entrada_procesada
    if saturacion_on:
        salida_temp = np.tanh(salida_temp) # No lineal

    # ESTABILIDAD BIBO: Añadimos feedback (y[n] depende de y[n-1])
    if n > 0:
        y[n] = salida_temp + (feedback_factor * y[n-1])
    else:
        y[n] = salida_temp

# Visualización
plt.figure(figsize=(12, 6))
plt.plot(t, x, 'gray', alpha=0.5, label='Entrada x(t) [Usuario]')
plt.plot(t, y, 'blue', linewidth=2, label='Salida y(t) [Antena]')
plt.fill_between(t, y, color='blue', alpha=0.1)
plt.axhline(0, color='black', lw=1)
plt.title(f"Respuesta del Sistema (Feedback={feedback_factor})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-1, 5 if feedback_factor < 1 else 20) # Ajuste de escala para ver la inestabilidad
plt.show()